# 02. Formal Neural Verification: IBP and CROWN Bound Propagation

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Gowal et al. (2018 *Effectiveness of Interval Bound Propagation*); Zhang et al. (2018 *Efficient Neural Network Robustness Certification with General Activation Functions: CROWN*); Xu et al. (2020 *Automatic Perturbation Analysis for Scalable Certified Robustness: auto_LiRPA*); Wang et al. (2021 *Beta-CROWN: Efficient Bound Propagation with Branch and Bound*).
- **Purpose:** Formulate and evaluate deterministic neural verification techniques: Interval Bound Propagation (IBP) and linear relaxation bound propagation (CROWN) to certify output margins and safety specifications under bounded input perturbation sets.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Adversarial Robustness FGSM PGD TRADES GCG and Smoothing](01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Machine Unlearning SISA Influence Scrub NPO and RMU](03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use pure IBP on deep standard networks without certified training (interval width explodes exponentially with depth); do not rely on incomplete convex relaxations alone when zero-tolerance verification is mandatory (use Branch-and-Bound).

## Mathematical Foundations & Governing Equations

1. **IBP Linear Layer**: $\mu_{\text{out}} = \mu W + b, \quad r_{\text{out}} = r |W|, \quad l = \mu_{\text{out}} - r_{\text{out}}, \quad u = \mu_{\text{out}} + r_{\text{out}}$
2. **IBP Monotonic Activation**: $l_{\text{out}} = \sigma(l_{\text{in}}), \quad u_{\text{out}} = \sigma(u_{\text{in}})$
3. **CROWN ReLU Relaxation**: $s_U = \frac{u}{u - l}, \quad b_U = -\frac{u \cdot l}{u - l}, \quad s_L = \mathbb{I}(u > |l|), \quad b_L = 0$
4. **CROWN Backward Margin Bound**: $\min_{x \in [x_l, x_u]} A_L x + \gamma_L = A_L \mu_x - |A_L| r_x + \gamma_L$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def ibp_linear(l_in, u_in, weight, bias):
    mu = (u_in + l_in) * 0.5
    r = (u_in - l_in) * 0.5
    mu_out = mu @ weight + bias
    r_out = r @ torch.abs(weight)
    return mu_out - r_out, mu_out + r_out

def ibp_relu(l_in, u_in):
    return torch.clamp(l_in, min=0.0), torch.clamp(u_in, min=0.0)

def crown_relu_relaxation(l_in, u_in):
    diff = torch.clamp(u_in - l_in, min=1e-8)
    s_u_cross = u_in / diff
    b_u_cross = -(u_in * l_in) / diff
    s_l_cross = torch.where(u_in > torch.abs(l_in), 1.0, 0.0)
    act = (l_in >= 0.0).float()
    inact = (u_in <= 0.0).float()
    cross = (1.0 - act) * (1.0 - inact)
    return act + cross * s_l_cross, torch.zeros_like(l_in), act + cross * s_u_cross, cross * b_u_cross

def crown_2layer(x_l, x_u, w1, b1, w2, b2, c):
    z1_l, z1_u = ibp_linear(x_l, x_u, w1, b1)
    s_L, b_L, s_U, b_U = crown_relu_relaxation(z1_l, z1_u)
    v = w2 @ c
    lam_l = torch.where(v >= 0.0, s_L, s_U) * v
    bias_l = torch.sum(torch.where(v >= 0.0, b_L, b_U) * v, dim=-1)
    a_l = lam_l @ w1.t()
    gam_l = torch.sum(lam_l * b1, dim=-1) + bias_l + torch.dot(b2, c)
    mu_x, r_x = (x_u + x_l) * 0.5, (x_u - x_l) * 0.5
    crown_l = torch.sum(a_l * mu_x, dim=-1) - torch.sum(torch.abs(a_l) * r_x, dim=-1) + gam_l
    lam_u = torch.where(v >= 0.0, s_U, s_L) * v
    bias_u = torch.sum(torch.where(v >= 0.0, b_U, b_L) * v, dim=-1)
    a_u = lam_u @ w1.t()
    gam_u = torch.sum(lam_u * b1, dim=-1) + bias_u + torch.dot(b2, c)
    crown_u = torch.sum(a_u * mu_x, dim=-1) + torch.sum(torch.abs(a_u) * r_x, dim=-1) + gam_u
    return crown_l, crown_u

In [ ]:
# Parity verification against golden reference tensors
z1_l, z1_u = ibp_linear(weights["x_l"], weights["x_u"], weights["w1"], weights["b1"])
assert torch.max(torch.abs(z1_l - weights["z1_l_ref"])) < 1e-5
assert torch.max(torch.abs(z1_u - weights["z1_u_ref"])) < 1e-5

a1_l, a1_u = ibp_relu(z1_l, z1_u)
assert torch.max(torch.abs(a1_l - weights["a1_l_ref"])) < 1e-5
assert torch.max(torch.abs(a1_u - weights["a1_u_ref"])) < 1e-5

s_L, b_L, s_U, b_U = crown_relu_relaxation(z1_l, z1_u)
assert torch.max(torch.abs(s_L - weights["s_L_ref"])) < 1e-5
assert torch.max(torch.abs(b_U - weights["b_U_ref"])) < 1e-5

cr_l, cr_u = crown_2layer(weights["x_l"], weights["x_u"], weights["w1"], weights["b1"], weights["w2"], weights["b2"], weights["c"])
assert torch.max(torch.abs(cr_l - weights["crown_lower_ref"])) < 1e-5
assert torch.max(torch.abs(cr_u - weights["crown_upper_ref"])) < 1e-5
print("All IBP and CROWN verification assertions passed!")

In [ ]:
# Benchmarking bound propagation latency
t0 = time.perf_counter()
for _ in range(500):
    _ = ibp_linear(weights["x_l"], weights["x_u"], weights["w1"], weights["b1"])
    _ = crown_2layer(weights["x_l"], weights["x_u"], weights["w1"], weights["b1"], weights["w2"], weights["b2"], weights["c"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined IBP + CROWN bound propagation latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Pure IBP Over-Approximation vs CROWN Tight Linear Relaxation
# Pure IBP propagates through intermediate activation boxes and loses inter-neuron correlations
z2_l, z2_u = ibp_linear(a1_l, a1_u, weights["w2"], weights["b2"])
c_vec = weights["c"]
c_pos = torch.clamp(c_vec, min=0.0)
c_neg = torch.clamp(c_vec, max=0.0)
ibp_margin_lower = (z2_l @ c_pos + z2_u @ c_neg)
crown_margin_lower = cr_l

print(f"Sample 0 Margin Lower Bound: IBP = {ibp_margin_lower[0].item():.4f} vs CROWN = {crown_margin_lower[0].item():.4f}")
assert (crown_margin_lower >= ibp_margin_lower - 1e-5).all(), "CROWN lower bound must be >= IBP lower bound!"
tightness_gain = (crown_margin_lower - ibp_margin_lower).mean().item()
print(f"CROWN average tightness advantage over IBP: +{tightness_gain:.4f}")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Interval Bound Propagation (IBP)** runs in $\mathcal{O}(L \cdot d^2)$ forward-pass time but accumulates loose over-approximation errors exponentially with depth.
2. **CROWN Linear Relaxation** computes backward linear inequalities that preserve correlation between neurons, producing provably tighter verified lower bounds.
3. **Complete Certification** combines fast GPU linear relaxation bound propagation with Branch-and-Bound (BaB) on crossing neurons.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_formal_neural_verification_ibp_and_crown_bound_propagation.md](../../markdown/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.md)
- **PyTorch 2.x Module:** [02_formal_neural_verification_ibp_and_crown_bound_propagation.py](../../pytorch2/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.py)
- **Functional JAX Module:** [02_formal_neural_verification_ibp_and_crown_bound_propagation.py](../../jax/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.py)
- **Native Rust Candle Track:** [02_formal_neural_verification_ibp_and_crown_bound_propagation.rs](../../candle/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.rs)
- **Hardware Kernel (CPU / WGSL):** [02_formal_neural_verification_ibp_and_crown_bound_propagation.rs](../../kernels/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.rs)
- **Golden SafeTensors:** [02_formal_neural_verification_ibp_and_crown_bound_propagation.safetensors](../../weights/11_ai_safety_privacy_and_verification/02_formal_neural_verification_ibp_and_crown_bound_propagation.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Adversarial Robustness FGSM PGD TRADES GCG and Smoothing](01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 03. Machine Unlearning SISA Influence Scrub NPO and RMU](03_machine_unlearning_sisa_influence_scrub_npo_and_rmu.ipynb) &rarr;